# Etapa 6 — Métricas de equidade

Projeto SME0828 — Introdução à Ciência de Dados, ICMC/USP.

Semente do código: 444. Reexecute o notebook para atualizar os resultados exibidos.


O notebook `05_modelagem_h2` treinou modelos com e sem a variável de gênero. Aqui comparamos as decisões previstas com as decisões reais dos bancos usando três métricas: diferença de paridade demográfica, disparate impact e diferença condicional a faixas de DTI e LTV. Calculamos intervalos de confiança por bootstrap.

### Bibliotecas

In [ ]:
from pathlib import Path
import os

ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents) if (path / "data").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Abra este notebook dentro do reposit?rio, que deve conter a pasta data/")
os.chdir(ROOT)

import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEMENTE = 444
random.seed(SEMENTE)
np.random.seed(SEMENTE)

BLUE   = "#2a78d6"
ORANGE = "#eb6834"
GRAY   = "#898781"

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)

### 2. Leitura das previsões

O notebook `05_modelagem_h2` salvou as previsões dos 459.866 pedidos do teste. O arquivo inclui a decisão real (`negado`), as probabilidades (`proba_com_genero`, `proba_sem_genero`) e as classes previstas (`pred_com_genero`, `pred_sem_genero`).

In [ ]:
data = pd.read_parquet("resultados/etapa_05_h2_predicoes_teste.parquet")

print("Número de linhas e colunas:", data.shape)
print(f"Mulheres no teste: {data['mulher'].mean():.1%}")
data.head()

Reunimos as decisões dos bancos e as previsões dos dois modelos em uma tabela.

In [ ]:
decisoes = {
    "Bancos (real)": data["negado"].values,
    "GB com gênero": data["pred_com_genero"].values,
    "GB sem gênero": data["pred_sem_genero"].values,
}
mulher = data["mulher"].values == 1

taxas = pd.DataFrame({
    nome: {"Homens": d[~mulher].mean(), "Mulheres": d[mulher].mean()}
    for nome, d in decisoes.items()
}).T
(100 * taxas).round(2)

,Homens,Mulheres
Bancos (real),9.93,11.34
GB com gênero,9.11,12.53
GB sem gênero,9.03,12.69


### 3. Métricas

Seja `ŷ = 1` uma negação prevista, `F` mulheres e `M` homens.

A diferença de paridade demográfica é `P(ŷ=1 | F) − P(ŷ=1 | M)`. Zero indica taxas iguais; valores positivos indicam mais negações previstas para mulheres.

O disparate impact compara as taxas de aprovação: `P(ŷ=0 | F) / P(ŷ=0 | M)`. Um indica igualdade; valores abaixo de 0,8 ficam abaixo da referência da regra dos 80%.

A diferença condicional compara mulheres e homens dentro de faixas de DTI e LTV, depois calcula a média ponderada dos estratos.

Dividimos o LTV em cinco quintis e cruzamos com as seis faixas de DTI.

In [ ]:
data["faixa_ltv"] = pd.qcut(data["ltv"], q=5, duplicates="drop")
data["estrato"] = data["dti"].astype(str) + " | " + data["faixa_ltv"].astype(str)
estrato = data["estrato"].values

print("Número de estratos:", data["estrato"].nunique())
data["estrato"].value_counts().head()

Número de estratos: 30


estrato
36%-<50% | (89.5, 95.0]       59304
36%-<50% | (79.78, 89.5]      52038
36%-<50% | (95.0, 97.0]       50942
36%-<50% | (13.987, 79.78]    49337
36%-<50% | (97.0, 115.52]     49204
Name: count, dtype: int64

Calculamos as três métricas para cada conjunto de decisões. Para a diferença condicional, calculamos primeiro a diferença em cada estrato e depois a média ponderada.

In [ ]:
resultados = []

for nome, d in decisoes.items():
    taxa_f = d[mulher].mean()
    taxa_m = d[~mulher].mean()

    por_estrato = pd.DataFrame({"estrato": estrato, "mulher": mulher, "d": d}).groupby(
        ["estrato", "mulher"])["d"].mean().unstack()
    tamanho = pd.Series(estrato).value_counts()
    dif_estrato = (por_estrato[True] - por_estrato[False]).dropna()
    condicional = np.average(dif_estrato, weights=tamanho[dif_estrato.index])

    resultados.append({
        "Decisão": nome,
        "Taxa homens (%)": 100 * taxa_m,
        "Taxa mulheres (%)": 100 * taxa_f,
        "DP (p.p.)": 100 * (taxa_f - taxa_m),
        "DI": (1 - taxa_f) / (1 - taxa_m),
        "DP condicional (p.p.)": 100 * condicional,
    })

metricas = pd.DataFrame(resultados).set_index("Decisão")
metricas.round(3)

,Taxa homens (%),Taxa mulheres (%),DP (p.p.),DI,DP condicional (p.p.)
Decisão,,,,,
Bancos (real),9.932,11.340,1.408,0.984,1.159
GB com gênero,9.108,12.533,3.425,0.962,3.475
GB sem gênero,9.031,12.687,3.656,0.960,3.717


### 4. Intervalos de confiança

Usamos bootstrap no conjunto de teste: sorteamos 459.866 pedidos com reposição, recalculamos as métricas e repetimos o processo mil vezes. O intervalo de 95% vai do percentil 2,5 ao 97,5.

O bootstrap cobre DP e DI. Não calculamos a métrica condicional em cada reamostragem, pois o agrupamento torna o cálculo muito mais lento.

In [ ]:
B = 1000
n = len(data)
boot = {nome: {"DP": [], "DI": []} for nome in decisoes}

for b in range(B):
    idx = np.random.randint(0, n, n)
    m = mulher[idx]
    for nome, d in decisoes.items():
        db = d[idx]
        taxa_f = db[m].mean()
        taxa_m = db[~m].mean()
        boot[nome]["DP"].append(100 * (taxa_f - taxa_m))
        boot[nome]["DI"].append((1 - taxa_f) / (1 - taxa_m))

for nome in decisoes:
    for metrica in ["DP", "DI"]:
        valores = np.array(boot[nome][metrica])
        metricas.loc[nome, f"{metrica} IC inf"] = np.percentile(valores, 2.5)
        metricas.loc[nome, f"{metrica} IC sup"] = np.percentile(valores, 97.5)

metricas[["DP (p.p.)", "DP IC inf", "DP IC sup", "DI", "DI IC inf", "DI IC sup", "DP condicional (p.p.)"]].round(3)

,DP (p.p.),DP IC inf,DP IC sup,DI,DI IC inf,DI IC sup,DP condicional (p.p.)
Decisão,,,,,,,
Bancos (real),1.408,1.223,1.586,0.984,0.982,0.986,1.159
GB com gênero,3.425,3.237,3.607,0.962,0.960,0.964,3.475
GB sem gênero,3.656,3.459,3.837,0.960,0.958,0.962,3.717


### 5. Visualização

O gráfico mostra a diferença de paridade e o disparate impact, ambos com intervalos de confiança de 95%. A linha de referência marca DI = 0,8.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
nomes = list(metricas.index)
y = np.arange(len(nomes))[::-1]
cores = [GRAY, ORANGE, BLUE]

axes[0].barh(y, metricas["DP (p.p.)"], color=cores, alpha=0.85)
axes[0].errorbar(metricas["DP (p.p.)"], y,
                 xerr=[metricas["DP (p.p.)"] - metricas["DP IC inf"], metricas["DP IC sup"] - metricas["DP (p.p.)"]],
                 fmt="none", ecolor="black", capsize=4)
axes[0].axvline(0, color="black", linewidth=0.8)
axes[0].set_xlabel("Mulheres − homens (p.p.)")
axes[0].set_title("Paridade demográfica", fontsize=13, fontweight="bold")

axes[1].barh(y, metricas["DI"], color=cores, alpha=0.85)
axes[1].axvline(0.8, color=ORANGE, linestyle="--", linewidth=1.5, label="Regra dos 80%")
axes[1].axvline(1.0, color="black", linewidth=0.8)
axes[1].set_xlim(0.7, 1.02)
axes[1].set_xlabel("Aprovação mulheres / aprovação homens")
axes[1].set_title("Disparate impact", fontsize=13, fontweight="bold")
axes[1].legend(frameon=False, loc="lower left")

for ax in axes:
    ax.set_yticks(y)
    ax.set_yticklabels(nomes)
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.savefig("figuras/etapa_06_equidade_dp_di.png", dpi=300)
plt.show()

### 6. Probabilidades e decisões

O modelo estima uma probabilidade e aplica um limiar para decidir entre aprovação e negação. Comparamos a diferença entre mulheres e homens nas duas etapas para verificar quanto o limiar altera o resultado.

In [ ]:
etapas = pd.DataFrame({
    "Bancos (real)": [100 * (data["negado"][mulher].mean() - data["negado"][~mulher].mean()), np.nan],
    "GB com gênero": [100 * (data["proba_com_genero"][mulher].mean() - data["proba_com_genero"][~mulher].mean()),
                      metricas.loc["GB com gênero", "DP (p.p.)"]],
    "GB sem gênero": [100 * (data["proba_sem_genero"][mulher].mean() - data["proba_sem_genero"][~mulher].mean()),
                      metricas.loc["GB sem gênero", "DP (p.p.)"]],
}, index=["Diferença nas probabilidades (p.p.)", "Diferença nas decisões (p.p.)"]).T
etapas.round(2)

,Diferença nas probabilidades (p.p.),Diferença nas decisões (p.p.)
Bancos (real),1.41,NaN
GB com gênero,1.47,3.43
GB sem gênero,1.65,3.66


A distribuição mostra as probabilidades previstas pelo modelo sem gênero para cada grupo. Valores à direita do limiar são classificados como negação.

In [ ]:
limiar = data.loc[data["pred_sem_genero"] == 1, "proba_sem_genero"].min()

fig, ax = plt.subplots(figsize=(9, 4.5))
bins = np.linspace(0, 0.6, 61)
ax.hist(data.loc[~mulher, "proba_sem_genero"], bins=bins, density=True, alpha=0.6, color=BLUE, label="Homens")
ax.hist(data.loc[mulher, "proba_sem_genero"], bins=bins, density=True, alpha=0.6, color=ORANGE, label="Mulheres")
ax.axvline(limiar, color="black", linestyle="--", linewidth=1.2, label=f"Limiar ({limiar:.3f})")
ax.set_xlabel("Probabilidade de negação prevista (GB sem gênero)")
ax.set_ylabel("Densidade")
ax.set_title("Distribuição das probabilidades por gênero", fontsize=13, fontweight="bold")
ax.legend(frameon=False)
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.savefig("figuras/etapa_06_equidade_distribuicao_probabilidades.png", dpi=300)
plt.show()

print(f"Probabilidade mediana - homens: {data.loc[~mulher, 'proba_sem_genero'].median():.3f}")
print(f"Probabilidade mediana - mulheres: {data.loc[mulher, 'proba_sem_genero'].median():.3f}")

Probabilidade mediana - homens: 0.060
Probabilidade mediana - mulheres: 0.065


### 7. Diferença por estrato de risco

O mapa mostra a diferença entre as decisões previstas para mulheres e homens em cada combinação de DTI e LTV. Laranja indica mais negações previstas para mulheres; azul, menos.

In [ ]:
tabela_estratos = data.groupby(["dti", "faixa_ltv", "mulher"], observed=True)["pred_sem_genero"].mean().unstack()
mapa = (100 * (tabela_estratos[1] - tabela_estratos[0])).unstack()

# Coloca as faixas de DTI na ordem do menor para o maior risco
ordem_dti = ["<20%", "20%-<30%", "30%-<36%", "36%-<50%", "50%-60%", ">60%"]
mapa = mapa.reindex(ordem_dti)

fig, ax = plt.subplots(figsize=(9, 5))
limite = np.nanmax(np.abs(mapa.values))
im = ax.imshow(mapa.values, cmap="RdBu_r", vmin=-limite, vmax=limite, aspect="auto")
ax.set_xticks(range(mapa.shape[1]))
ax.set_xticklabels([str(c) for c in mapa.columns], rotation=30, ha="right")
ax.set_yticks(range(mapa.shape[0]))
ax.set_yticklabels(mapa.index)
for i in range(mapa.shape[0]):
    for j in range(mapa.shape[1]):
        if not np.isnan(mapa.values[i, j]):
            ax.text(j, i, f"{mapa.values[i, j]:.1f}", ha="center", va="center", fontsize=8)
ax.set_xlabel("Faixa de LTV")
ax.set_ylabel("Faixa de DTI")
ax.set_title("Mulheres − homens: taxa de negação prevista (p.p.)", fontsize=13, fontweight="bold")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig("figuras/etapa_06_equidade_estratos.png", dpi=300)
plt.show()

### 8. Métricas que não calculamos

O HMDA público não informa inadimplência. Por isso, não podemos calcular equalized odds, equal opportunity ou calibração por grupo com o desfecho real. Usar a decisão do banco como rótulo mediria apenas o quanto o modelo reproduz essa decisão, não se ele é justo.

### 9. Salvamento dos resultados

In [ ]:
tabela_final = metricas.copy()
tabela_final["DP (IC 95%)"] = tabela_final.apply(lambda r: f"{r['DP (p.p.)']:.2f} ({r['DP IC inf']:.2f}; {r['DP IC sup']:.2f})", axis=1)
tabela_final["DI (IC 95%)"] = tabela_final.apply(lambda r: f"{r['DI']:.3f} ({r['DI IC inf']:.3f}; {r['DI IC sup']:.3f})", axis=1)
tabela_final["DP condicional"] = tabela_final["DP condicional (p.p.)"].map(lambda x: f"{x:.2f}")
tabela_final = tabela_final[["Taxa homens (%)", "Taxa mulheres (%)", "DP (IC 95%)", "DI (IC 95%)", "DP condicional"]].round(2)

with open("resultados/etapa_06_equidade_metricas.md", "w", encoding="utf-8") as f:
    f.write(tabela_final.to_markdown())
    f.write("\n\nDP em pontos percentuais (mulheres − homens) na taxa de negação. DI = aprovação mulheres / aprovação homens "
            "(regra dos 80%: DI < 0,8). DP condicional: média ponderada da DP em 30 estratos de DTI × quintil de LTV. "
            "IC 95% por bootstrap (1.000 reamostragens do conjunto de teste, n = 459.866).\n")
tabela_final

,Taxa homens (%),Taxa mulheres (%),DP (IC 95%),DI (IC 95%),DP condicional
Decisão,,,,,
Bancos (real),9.93,11.34,1.41 (1.22; 1.59),0.984 (0.982; 0.986),1.16
GB com gênero,9.11,12.53,3.43 (3.24; 3.61),0.962 (0.960; 0.964),3.48
GB sem gênero,9.03,12.69,3.66 (3.46; 3.84),0.960 (0.958; 0.962),3.72


### Conclusão

- Os bancos negam 1,41 p.p. mais pedidos de mulheres (IC 95%: 1,22 a 1,59). Entre pedidos com DTI e LTV semelhantes, a diferença cai para 1,16 p.p.
- Os modelos preveem uma diferença maior: 3,43 p.p. com gênero e 3,66 p.p. sem gênero (IC 95%: 3,46 a 3,84).
- Remover gênero não elimina a disparidade. As demais variáveis mantêm a diferença nas previsões.
- No modelo sem gênero, a diferença condicional é 3,72 p.p.; mulheres têm mais negações previstas em quase todos os estratos.
- A diferença entre as probabilidades é 1,65 p.p.; após aplicar o limiar de 0,143, a diferença entre as decisões sobe para 3,66 p.p.
- Os valores de DI ficam acima de 0,95, portanto acima da referência de 0,8 da regra dos 80%.
- Para DTI acima de 60%, o modelo prevê negação para todos os pedidos dos dois grupos.

A Etapa F investiga quais variáveis carregam informação associada ao gênero.